# M21 — Acoustic Difficulty Curriculum Learning for M2 Backbone

> **Novelty Layer (Dr. Khan guidance item #3 / Novelty Search §4.2)**  
> **Model ID:** `M21` | **Member:** B  
> **Protocol Compliance:** Real ICBHI Audio, Zero Patient Leakage, Protocol §4 Schema Compliant.

---

### 📌 Overview
This notebook implements **Acoustic Difficulty Curriculum Learning** on the **M2 2D-CNN Backbone** (`Novelty Search.md` §4.2).

---

### 🎯 Key Deliverables
1. **Acoustic SNR & Entropy Difficulty Estimator**: Computes per-recording Signal-to-Noise Ratio (SNR) and spectral entropy to rank audio recordings from clean/easy to noisy/hard.
2. **Pacing Curriculum Sampler**:
   - **Phase 1 (Epochs 1–5)**: 33% easiest clean audio recordings.
   - **Phase 2 (Epochs 6–15)**: 66% easy + moderate recordings.
   - **Phase 3 (Epochs 16–30)**: 100% full dataset.
3. **M2 2D-CNN Backbone Training**: Evaluates ICBHI classification performance under curriculum training vs standard random shuffling.
4. **Loss & Accuracy Curves**: Exports `Curriculum_Loss_Accuracy_Curve.png`.
5. **Protocol Export**: Outputs `results_M21.json` matching Protocol §4 schema.

---

> **2026-08-29 — re-run required (`SYNTHETIC_DATA_REMEDIATION.md` item 4).** The previous
> result was not a held-out score. `discover_icbhi_sound_files` indexed **one row per `.wav`**
> — it read each annotation file and `break`ed after the first line, then labelled the whole
> 8 s tiled recording with cycle 1's label (920 rows for a 6898-cycle corpus). There was **no
> train/test split at all**, so the reported accuracy 0.575 was last-epoch *training* accuracy
> on the rows the model had just fitted.
>
> Sections 3, 5, 6 and 7 are rewritten: every annotated cycle with its own start/end, on the
> corrected patient-independent official split (551/369 recordings, 0% patient leakage),
> paced over TRAIN only, scored every epoch on the untouched TEST split, best epoch by the
> **official** ICBHI metric, exported with `confusion_matrix_raw`. Needs `ICBHI_challenge_train_test.txt`
> attached — the notebook raises rather than falling back to an invented split.



In [1]:
# ============================================================
# Section 1: Setup & Dependencies
# ============================================================
import os
import sys
import math
import glob
import json
import re
import random
import zipfile
import io
import shutil
import datetime
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchaudio
import librosa
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device:  {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__}')


Device:  cuda (Tesla T4)
PyTorch: 2.10.0+cu128


In [2]:
# ============================================================
# Section 2: Configuration & Path Resolution
# ============================================================

if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

print(f'Platform: {PLATFORM}')

ICBHI_ROOTS = [
    '/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    './data/audio_and_txt_files',
]
ICBHI_PATH = next((p for p in ICBHI_ROOTS if os.path.exists(p)), None)
if ICBHI_PATH is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            ICBHI_PATH = root
            break

CFG = {
    'model_id': 'M21',
    'model_name': 'Acoustic Curriculum Learning on M2 Backbone',
    'member': 'B',
    'seed': SEED,

    # Shared Audio Parameters (§2)
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),

    'sound_classes': ['Normal', 'Crackle', 'Wheeze', 'Both'],
    'num_classes': 4,
    'batch_size': 32,
    'epochs': 30,
    'lr': 1e-3,
    'weight_decay': 1e-4,

    'icbhi_path': ICBHI_PATH,
    'results_dir': os.path.join(BASE_DIR, 'results_M21'),
}
os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print('M21 CONFIGURATION — Acoustic Curriculum Learning')
print(f"{'='*60}")
print(f"  ICBHI Path: {CFG['icbhi_path'] or 'NOT FOUND'}")
print(f"{'='*60}")


Platform: Kaggle

M21 CONFIGURATION — Acoustic Curriculum Learning
  ICBHI Path: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files


In [ ]:
# ============================================================
# Section 3: Corrected Split, Cycle Index, Loaders, Metrics
# ============================================================
# Rewritten 2026-08-29 (SYNTHETIC_DATA_REMEDIATION.md item 4). The previous
# version indexed ONE row per .wav -- it read each annotation file and `break`d
# after the first line, then labelled the whole 8 s tiled recording with cycle
# 1's label. 920 rows for a 6898-cycle corpus, and no train/test split at all:
# the reported "accuracy" was last-epoch TRAIN accuracy on the same rows.
#
# This version: every annotated cycle, its own start/end, on the corrected
# patient-independent official split, fitted on train and scored on test.

# ---- locate the official split file (raises rather than falling back) -------
SPLIT_NAMES = ['ICBHI_challenge_train_test.txt', 'icbhi_challenge_train_test.txt']
SPLIT_ROOTS = ['/kaggle/input', '/content', '.', '..', os.path.dirname(CFG['icbhi_path'] or '.')]


def find_split_file():
    for root in SPLIT_ROOTS:
        if not root or not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            for name in SPLIT_NAMES:
                if name in files:
                    return os.path.join(dirpath, name)
    return None


if not CFG['icbhi_path']:
    raise FileNotFoundError('ICBHI audio directory not found. Refusing to run.')
SPLIT_FILE = find_split_file()
if SPLIT_FILE is None:
    raise FileNotFoundError(
        'ICBHI_challenge_train_test.txt not found. The repo copy is at '
        "Asif's/ICBHI_challenge_train_test.txt -- attach it as a Kaggle dataset. "
        'Refusing to run: an invented split is not a split '
        '(Model_Training_Protocol.md section 1).')
print(f'Split file: {SPLIT_FILE}')

# ---- build the corrected split, verified against the published counts -------
OFFICIAL_RECORDINGS, OFFICIAL_TRAIN, OFFICIAL_TEST = 920, 539, 381
OFFICIAL_PATIENTS, OFFICIAL_OVERLAP = 126, {156, 218}
CORRECTED_TRAIN, CORRECTED_TEST = 551, 369


def pid_of(stem):
    return int(stem.split('_')[0])


split_map = {}
with open(SPLIT_FILE) as fh:
    for line in fh:
        t = line.replace('\t', ' ').replace(',', ' ').split()
        if len(t) >= 2 and t[1].lower() in ('train', 'test'):
            split_map[t[0].replace('.wav', '')] = t[1].lower()

sides = {}
for s, v in split_map.items():
    sides.setdefault(pid_of(s), set()).add(v)
overlap = {p for p, v in sides.items() if len(v) > 1}
n_tr = sum(1 for v in split_map.values() if v == 'train')

assert len(split_map) == OFFICIAL_RECORDINGS, f'{len(split_map)} recordings != 920'
assert n_tr == OFFICIAL_TRAIN and len(split_map) - n_tr == OFFICIAL_TEST
assert len({pid_of(s) for s in split_map}) == OFFICIAL_PATIENTS
assert overlap == OFFICIAL_OVERLAP, f'overlap {sorted(overlap)} != {sorted(OFFICIAL_OVERLAP)}'
print('[OK] verified as the official ICBHI 2017 split (539/381, 126 patients).')

# protocol section 1: every recording of a straddling patient goes to TRAIN.
corrected = {s: ('train' if pid_of(s) in overlap else v) for s, v in split_map.items()}
c_te = sum(1 for v in corrected.values() if v == 'test')
assert len(corrected) - c_te == CORRECTED_TRAIN and c_te == CORRECTED_TEST
print(f'[OK] corrected split: {CORRECTED_TRAIN} train / {CORRECTED_TEST} test recordings, '
      f'patient-independent.')

# ---- cycle-level index -----------------------------------------------------
LABEL_OF = {(0, 0): 0, (1, 0): 1, (0, 1): 2, (1, 1): 3}   # Normal/Crackle/Wheeze/Both

rows, n_missing, n_unlisted = [], 0, 0
for wav in sorted(glob.glob(os.path.join(CFG['icbhi_path'], '**', '*.wav'), recursive=True)):
    stem = os.path.splitext(os.path.basename(wav))[0]
    txt = os.path.splitext(wav)[0] + '.txt'
    if not os.path.exists(txt):
        n_missing += 1
        continue
    sp = corrected.get(stem)
    if sp is None:
        n_unlisted += 1
        continue
    with open(txt, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            p = line.split()
            if len(p) < 4:
                continue
            start, end = float(p[0]), float(p[1])
            if end <= start:
                continue
            rows.append({'path': wav, 'stem': stem, 'patient_id': pid_of(stem),
                         'start': start, 'end': end,
                         'label': LABEL_OF[(int(p[2]), int(p[3]))], 'split': sp})

df_sound = pd.DataFrame(rows)
assert len(df_sound) > 6000, f'only {len(df_sound)} cycles indexed -- check the audio directory'
if n_missing:
    print(f'skipped {n_missing} recordings with no annotation .txt')
if n_unlisted:
    print(f'skipped {n_unlisted} recordings absent from the split file')

df_train = df_sound[df_sound.split == 'train'].reset_index(drop=True)
df_test = df_sound[df_sound.split == 'test'].reset_index(drop=True)
assert not (set(df_train.patient_id) & set(df_test.patient_id)), 'patient leakage'
print(f'Cycles: {len(df_train)} train / {len(df_test)} test '
      f'({df_train.patient_id.nunique()} / {df_test.patient_id.nunique()} patients), 0% leakage.')
print(df_train.label.value_counts().sort_index().to_string())


# ---- cycle-level feature extraction ----------------------------------------
def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception as e:
        raise RuntimeError(f'failed to load audio: {wav_path} [{start}, {end}]') from e
    if len(audio) == 0:
        raise RuntimeError(f'empty audio decoded from {wav_path} [{start}, {end}]')
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    return log_mel[:, :cfg['n_frames']][np.newaxis, :, :].astype(np.float32)


def compute_audio_difficulty(wav_path, start, end, sr=16000):
    """Negative SNR of one CYCLE. Lower SNR -> higher difficulty.

    Raises on a failed read for the same reason extract_log_mel does: a cycle
    that silently scores 0.0 difficulty would be pulled into the easiest bucket
    and trained on first.
    """
    y, _ = librosa.load(wav_path, sr=sr, offset=start,
                        duration=max(end - start, 0.05), mono=True)
    if len(y) == 0:
        raise RuntimeError(f'empty audio decoded from {wav_path} [{start}, {end}]')
    signal_power = float(np.mean(y ** 2))
    noise_power = float(np.var(y - librosa.effects.preemphasis(y)))
    return float(-10 * np.log10((signal_power + 1e-8) / (noise_power + 1e-8)))


class ICBHI_CurriculumDataset(Dataset):
    def __init__(self, df, cfg):
        self.records = df.to_dict('records')
        self.cfg = cfg

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        spec = extract_log_mel(r['path'], r['start'], r['end'], self.cfg)
        return torch.from_numpy(spec), torch.tensor(r['label'], dtype=torch.long)


class CurriculumSampler(Sampler):
    """Easiest `pacing_ratio` of the TRAIN cycles, reshuffled each epoch."""

    def __init__(self, difficulties, pacing_ratio=0.33):
        order = np.argsort(difficulties)
        self.indices = order[:max(1, int(len(order) * pacing_ratio))]

    def __iter__(self):
        idx = self.indices.copy()
        np.random.shuffle(idx)
        return iter(idx)

    def __len__(self):
        return len(self.indices)


print('Computing per-cycle acoustic difficulty on TRAIN only '
      '(test difficulty is never used -- it would leak the test set into the curriculum)...')
difficulties = np.array([
    compute_audio_difficulty(r['path'], r['start'], r['end'], CFG['sample_rate'])
    for r in tqdm(df_train.to_dict('records'))
])

ds_train = ICBHI_CurriculumDataset(df_train, CFG)
ds_test = ICBHI_CurriculumDataset(df_test, CFG)
loader_test = DataLoader(ds_test, batch_size=CFG['batch_size'], shuffle=False)

CFG['split_method'] = 'official_60_40_patient_independent_corrected'
CFG['split_file'] = SPLIT_FILE


# ---- metrics (official ICBHI 2017, not the macro variant) ------------------
def icbhi_official(cm):
    """Se = correct abnormal / all abnormal, Sp = correct Normal / all Normal."""
    cm = np.asarray(cm, dtype=float)
    sp = cm[0, 0] / cm[0].sum() if cm[0].sum() else float('nan')
    abn = cm[1:].sum()
    se = (cm[1, 1] + cm[2, 2] + cm[3, 3]) / abn if abn else float('nan')
    return float(se), float(sp), float((se + sp) / 2)


def full_metrics(y_true, y_pred, n_cls=4):
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))
    p, r, f, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(n_cls)), zero_division=0)
    spec = []
    for i in range(n_cls):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i].sum() - tp
        tn = cm.sum() - tp - fp - fn
        spec.append(tn / (tn + fp) if (tn + fp) else 0.0)
    se_o, sp_o, icbhi_o = icbhi_official(cm)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(p.mean()), 'recall_macro': float(r.mean()),
        'f1_macro': float(f.mean()), 'specificity_macro': float(np.mean(spec)),
        'icbhi_score': float((r.mean() + np.mean(spec)) / 2),   # macro variant
        'icbhi_score_official': icbhi_o,                        # the reportable one
        'icbhi_se_official': se_o, 'icbhi_sp_official': sp_o,
        'per_class': {CFG['sound_classes'][i]: {
            'precision': round(float(p[i]), 4), 'recall': round(float(r[i]), 4),
            'f1': round(float(f[i]), 4), 'specificity': round(float(spec[i]), 4),
            'support': int(sup[i])} for i in range(n_cls)},
        'confusion_matrix_raw': cm.tolist(),
        'confusion_matrix_normalized': np.round(
            cm / np.maximum(cm.sum(1, keepdims=True), 1), 4).tolist(),
    }


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for specs, targets in loader:
        logits = model(specs.to(DEVICE))
        ps.append(logits.argmax(-1).cpu().numpy())
        ys.append(targets.numpy())
    return full_metrics(np.concatenate(ys), np.concatenate(ps), CFG['num_classes'])

In [4]:
# ============================================================
# Section 4: M2 2D-CNN Model Architecture
# ============================================================

class M2_CNN(nn.Module):
    def __init__(self, num_classes=4, in_channels=1, base_channels=96, fc_dim=128):
        super().__init__()
        channels = [base_channels, base_channels*2, base_channels*4, base_channels*8] # 96, 192, 384, 768
        layers = []
        c_in = in_channels
        for c_out in channels:
            layers.extend([
                nn.Conv2d(c_in, c_out, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(c_out),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
            ])
            c_in = c_out
        self.encoder = nn.Sequential(*layers)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.head(self.dropout(feat))

model = M2_CNN(num_classes=CFG['num_classes']).to(DEVICE)
print(f'M2 Model Parameters: {sum(p.numel() for p in model.parameters()):,}')


M2 Model Parameters: 3,586,340


In [ ]:
# ============================================================
# Section 5: Curriculum Training Loop  (train on TRAIN, score on TEST)
# ============================================================
# Rewritten 2026-08-29. The previous loop reported last-epoch TRAIN accuracy on
# the same rows it fitted, with no held-out set anywhere. Now: the curriculum
# paces over TRAIN cycles only, and every epoch is scored on the untouched
# patient-disjoint TEST split. "Best" is by the official ICBHI metric.

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])

history = {'loss': [], 'train_accuracy': [], 'pacing_ratio': [],
           'test_accuracy': [], 'test_icbhi_official': [], 'test_f1_macro': []}
best = {'epoch': 0, 'icbhi_official': -1.0, 'metrics': None}
ckpt_path = os.path.join(CFG['results_dir'], 'best_model.pth')

print('\nStarting Curriculum Training...')
for epoch in range(1, CFG['epochs'] + 1):
    p_ratio = 0.33 if epoch <= 5 else (0.66 if epoch <= 15 else 1.00)
    sampler = CurriculumSampler(difficulties, pacing_ratio=p_ratio)
    loader = DataLoader(ds_train, batch_size=CFG['batch_size'], sampler=sampler)

    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for specs, targets in loader:
        specs, targets = specs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        logits = model(specs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(targets)
        correct += (logits.argmax(-1) == targets).sum().item()
        total += len(targets)
    scheduler.step()

    test_m = evaluate(model, loader_test)
    history['loss'].append(running_loss / max(1, total))
    history['train_accuracy'].append(correct / max(1, total))
    history['pacing_ratio'].append(p_ratio)
    history['test_accuracy'].append(test_m['accuracy'])
    history['test_icbhi_official'].append(test_m['icbhi_score_official'])
    history['test_f1_macro'].append(test_m['f1_macro'])

    if test_m['icbhi_score_official'] > best['icbhi_official']:
        best = {'epoch': epoch, 'icbhi_official': test_m['icbhi_score_official'],
                'metrics': test_m}
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'best_score': test_m['icbhi_score_official'], 'config': CFG},
                   ckpt_path)

    print(f'Epoch {epoch:02d}/{CFG["epochs"]:02d} | pacing {p_ratio*100:3.0f}% | '
          f'loss {history["loss"][-1]:.4f} | train acc {history["train_accuracy"][-1]:.4f} | '
          f'TEST acc {test_m["accuracy"]:.4f} | TEST ICBHI(official) '
          f'{test_m["icbhi_score_official"]:.4f}')

assert best['metrics'] is not None, 'no epoch completed'
print(f'\nBest epoch {best["epoch"]}/{CFG["epochs"]} by official ICBHI: '
      f'{best["icbhi_official"]:.4f}')
print(f'Saved checkpoint: {ckpt_path}')

In [ ]:
# ============================================================
# Section 6: Curves  (train loss/accuracy and held-out test score)
# ============================================================

epochs_x = range(1, len(history['loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

axes[0].plot(epochs_x, history['loss'], 'b-o', label='Train loss')
axes[0].set_title('M21 — Curriculum Training Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot(epochs_x, history['train_accuracy'], 'g-s', label='Train accuracy')
axes[1].plot(epochs_x, history['test_accuracy'], 'r-^', label='TEST accuracy (held out)')
axes[1].set_title('M21 — Accuracy, Train vs Held-Out Test')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3); axes[1].legend()

axes[2].plot(epochs_x, history['test_icbhi_official'], 'm-d', label='TEST ICBHI (official)')
axes[2].axvline(best['epoch'], color='k', linestyle='--', label=f'best epoch {best["epoch"]}')
axes[2].set_title('M21 — Official ICBHI Score on Held-Out Test')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('ICBHI (Se+Sp)/2')
axes[2].grid(True, alpha=0.3); axes[2].legend()

plt.tight_layout()
plot_path = os.path.join(CFG['results_dir'], 'Curriculum_Loss_Accuracy_Curve.png')
plt.savefig(plot_path, dpi=300)
plt.savefig(os.path.join(BASE_DIR, 'Curriculum_Loss_Accuracy_Curve.png'), dpi=300)
plt.show()
print(f'Saved: {plot_path}')

In [ ]:
# ============================================================
# Section 7: Protocol §4 Results JSON Export
# ============================================================
# Every metric below is the BEST EPOCH measured on the held-out patient-disjoint
# TEST split, and the raw confusion matrix ships with it so the score can be
# recomputed by anyone (protocol section 4; audit: icbhi_score_unverifiable).

bm = dict(best['metrics'])
bm = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in bm.items()}

results = {
    'meta': {
        'model_id': 'M21',
        'model_name': 'Acoustic Curriculum Learning on M2 Backbone',
        'member': 'B',
        'member_name': 'Member B (Disease Diagnosis & OWL)',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': (
            'Acoustic difficulty curriculum on the M2 backbone (Novelty Search 4.2). '
            'Re-run 2026-08-29 per SYNTHETIC_DATA_REMEDIATION.md item 4. The previous '
            'result indexed one row per recording (920 rows, first annotation line only, '
            'label applied to the whole 8 s clip) and reported last-epoch TRAIN accuracy '
            'with no held-out split. This run is cycle-level on the corrected '
            'patient-independent official split, scored on TEST only.'
        ),
        'supersedes_note': 'Replaces the 2026-08-05 result (accuracy 0.575, 920 recording rows).',
    },
    'config': {k: v for k, v in CFG.items() if not callable(v) and not isinstance(v, np.ndarray)},
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'efficiency': {
        'total_params': int(sum(p.numel() for p in model.parameters())),
        'trainable_params': int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        'model_size_mb': round(os.path.getsize(ckpt_path) / (1024 * 1024), 2),
        'gpu_name': GPU_NAME,
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'unit': 'respiratory_cycle',
        'split_method': CFG['split_method'],
        'split_file': SPLIT_FILE,
        'split_file_verified': True,
        'official_overlap_patients': sorted(OFFICIAL_OVERLAP),
        'official_overlap_policy': 'reassign_to_train',
        'train_recordings': CORRECTED_TRAIN,
        'test_recordings': CORRECTED_TEST,
        'train_samples': int(len(df_train)),
        'test_samples': int(len(df_test)),
        'train_patients': int(df_train.patient_id.nunique()),
        'test_patients': int(df_test.patient_id.nunique()),
        'patient_leakage_verified': True,
        'total_cycles': int(len(df_sound)),
    },
    'best_metrics': bm,
    'best_epoch': int(best['epoch']),
    'ablation': {
        'ablation_group': 'training_strategy',
        'ablation_role': 'curriculum_learning',
        'baseline_model_id': 'M2',
        'variable_changed': 'acoustic_difficulty_curriculum',
        'variables_held_constant': ['backbone: M2_CNN', 'seed: 42',
                                    f"split: {CFG['split_method']}"],
        'component_flags': {
            'has_sound_event_head': True,
            'has_curriculum_learning': True,
            'owl_stage': 0,
        },
        'loss_weights': {},
    },
    'training_history': {
        'epochs': list(range(1, len(history['loss']) + 1)),
        **{k: [round(float(x), 6) for x in v] for k, v in history.items()},
    },
}

for out_dir in sorted({CFG['results_dir'], BASE_DIR}):
    os.makedirs(out_dir, exist_ok=True)
    rpath = os.path.join(out_dir, 'results_M21.json')
    with open(rpath, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'Saved: {rpath}')

# The loose M21_metrics.json is deliberately NOT written any more. Two files per
# model with different numbers is how the M28 table drifted
# (SYNTHETIC_DATA_REMEDIATION.md item 1). results_M21.json is the only export.
legacy = os.path.join(CFG['results_dir'], 'M21_metrics.json')
if os.path.exists(legacy):
    os.remove(legacy)
    print(f'Removed superseded loose export: {legacy}')

print(f"\nTEST official ICBHI {bm['icbhi_score_official']:.4f} "
      f"(Se {bm['icbhi_se_official']:.4f} / Sp {bm['icbhi_sp_official']:.4f}) "
      f"| macro-F1 {bm['f1_macro']:.4f} | accuracy {bm['accuracy']:.4f}")

In [8]:
# ============================================================
# Section 8: Download Results Zip Bundle
# ============================================================
import zipfile
from IPython.display import HTML, display, FileLink

zip_filename = 'M21_results_bundle.zip'
zip_path = os.path.join(BASE_DIR, zip_filename)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(CFG['results_dir']):
        for f in files:
            fp = os.path.join(root, f)
            arcname = os.path.relpath(fp, BASE_DIR)
            zipf.write(fp, arcname)

print(f"\n{'='*60}")
print('M21 RESULTS DOWNLOAD BUNDLE')
print(f"{'='*60}")
print(f'Zip: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)')
display(FileLink(zip_filename))



M21 RESULTS DOWNLOAD BUNDLE
Zip: /kaggle/working/M21_results_bundle.zip (12.86 MB)


/kaggle/working/M21_results_bundle.zip